In [1]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("sentiment_data.csv")
df = df.dropna(subset=["text", "sentiment"])

# Encode sentiment labels
mapping = {"negative": 0, "neutral": 1, "positive": 2}
reverse = {0: "Negative", 1: "Neutral", 2: "Positive"}

df["sentiment"] = df["sentiment"].str.lower().str.strip()
df["label"] = df["sentiment"].map(mapping)
df = df.dropna(subset=["label"])

X = df["text"].astype(str)
y = df["label"].astype(int)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Create models
ada = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("model", AdaBoostClassifier(n_estimators=100, random_state=42))
])

rf = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("model", RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])

xgb = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("model", XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        objective="multi:softmax", num_class=3,
        eval_metric="mlogloss", random_state=42
    ))
])

# Train models
ada.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# Calculate accuracy
ada_acc = accuracy_score(y_test, ada.predict(X_test))
rf_acc = accuracy_score(y_test, rf.predict(X_test))
xgb_acc = accuracy_score(y_test, xgb.predict(X_test))

print("=" * 50)
print("MODEL ACCURACY")
print("=" * 50)
print(f"AdaBoost      : {ada_acc * 100:.2f}%")
print(f"Random Forest : {rf_acc * 100:.2f}%")
print(f"XGBoost       : {xgb_acc * 100:.2f}%")

# Real-time sentiment prediction
print("\n" + "=" * 50)
print("REAL-TIME SENTIMENT ANALYSIS")
print("=" * 50)
print("Type 'exit' to stop.")

while True:
    review = input("\nEnter your review: ")

    if review.lower().strip() == "exit":
        print("Program ended.")
        break

    if not review.strip():
        print("Please enter a valid review.")
        continue

    print("\nSentiment:")
    print("AdaBoost      :", reverse[int(ada.predict([review])[0])])
    print("Random Forest :", reverse[int(rf.predict([review])[0])])
    print("XGBoost       :", reverse[int(xgb.predict([review])[0])])


MODEL ACCURACY
AdaBoost      : 66.75%
Random Forest : 97.68%
XGBoost       : 97.63%

REAL-TIME SENTIMENT ANALYSIS
Type 'exit' to stop.

Sentiment:
AdaBoost      : Positive
Random Forest : Negative
XGBoost       : Neutral
Program ended.
